代码示例见目录 `10_claudecode_hooks`. 此处仅为笔记

hooks解决3个问题：

1. 权限管理。可以制定一个权限策略：高频且安全的操作自动放行，低频且危险的操作必须拦截或人工确认
2. 审计追溯。在多 Agent、多 Session、多 Fork 的生产环境中，工具调用链会迅速变得复杂。若无自动审计，事后复盘复杂或不可能。每一次工具调用的输入、输出、调用时间、调用者、所属 Session 等信息，并按 session_id 或分支组织起来。
3. 观测subagent。 感知subagent的启停事件。


[claude agent sdk](https://code.claude.com/docs/en/agent-sdk/hooks)

| Hook Event | Python SDK | TypeScript SDK | What triggers it | Example use case |
| --- | --- | --- | --- | --- |
| **PreToolUse** | Yes | Yes | Tool call request (can block or modify) | Block dangerous shell commands |
| **PostToolUse** | Yes | Yes | Tool execution result | Log all file changes to audit trail |
| **PostToolUseFailure** | Yes | Yes | Tool execution failure | Handle or log tool errors |
| **PostToolBatch** | No | Yes | A full batch of tool calls resolves, once per batch before the next model call | Inject conventions once for the whole batch |
| **UserPromptSubmit** | Yes | Yes | User prompt submission | Inject additional context into prompts |
| **UserPromptExpansion** | No | Yes | A user-typed command, or an MCP prompt, expands into a prompt before it reaches Claude. Doesn’t fire when Claude invokes a skill itself | Block a command from direct invocation or add context when a skill is typed |
| **MessageDisplay** | No | Yes | An assistant message with text completes, once per message with the full message text | Redact or reformat the displayed text without changing the transcript |
| **Stop** | Yes | Yes | Agent execution stop | Save session state before exit |
| **StopFailure** | No | Yes | The turn ends with an API error instead of a normal stop | Log failures or send alerts |
| **SubagentStart** | Yes | Yes | Subagent initialization | Track parallel task spawning |
| **SubagentStop** | Yes | Yes | Subagent completion | Aggregate results from parallel tasks |
| **PreCompact** | Yes | Yes | Conversation compaction request | Archive full transcript before summarizing |
| **PostCompact** | No | Yes | Conversation compaction completes | Log the generated summary |
| **PermissionRequest** | Yes | Yes | A tool call needs a permission decision | Custom permission handling |
| **PermissionDenied** | No | Yes | The auto mode classifier denies a tool call | Log classifier denials or tell the model it may retry |
| **SessionStart** | No | Yes | Session initialization | Initialize logging and telemetry |
| **SessionEnd** | No | Yes | Session termination | Clean up temporary resources |
| **Notification** | Yes | Yes | Agent status messages | Send agent status updates to Slack or PagerDuty |
| **Setup** | No | Yes | Session setup/maintenance | Run initialization tasks |
| **TeammateIdle** | No | Yes | Teammate becomes idle | Reassign work or notify |
| **TaskCreated** | No | Yes | A task is created via the `TaskCreate` tool | Enforce task naming conventions |
| **TaskCompleted** | No | Yes | Background task completes | Aggregate results from parallel tasks |
| **Elicitation** | No | Yes | An MCP server requests user input mid-task | Respond to MCP input requests programmatically |
| **ElicitationResult** | No | Yes | A user responds to an MCP elicitation | Modify or block the response before it returns to the server |
| **ConfigChange** | No | Yes | Configuration file changes | Reload settings dynamically |
| **InstructionsLoaded** | No | Yes | A `CLAUDE.md` or rules file is loaded into context | Audit which instruction files load |
| **WorktreeCreate** | No | Yes | Git worktree created | Track isolated workspaces |
| **WorktreeRemove** | No | Yes | Git worktree removed | Clean up workspace resources |
| **CwdChanged** | No | Yes | The working directory changes during a session | Reload environment variables per directory |
| **FileChanged** | No | Yes | A watched file is modified, created, or deleted | Reload configuration when project files change |
| **DirectoryAdded** | No | Yes | A working directory is added during a session | Install dependencies for a repository added mid-session |

回调签名固定是 (input_data, tool_use_id, context)

每个 Hook 回调都会接收三个参数：

* **Input data：** 一个包含事件详细信息的强类型对象。每种 Hook 类型都有各自的输入结构。例如，`PreToolUseHookInput` 包含 `tool_name` 和 `tool_input`，而 `NotificationHookInput` 包含 `message`。
    * 所有 Hook 输入都会共享 `session_id`、`cwd` 和 `hook_event_name`。
    * 当 Hook 在子 Agent（Subagent）内部触发时，会填充 `agent_id` 和 `agent_type`。在 TypeScript 中，它们位于基础 Hook 输入中，适用于所有 Hook 类型。在 Python 中，它们是 `PreToolUse`、`PostToolUse`、`PostToolUseFailure` 和 `PermissionRequest` 上的可选字段，并且是 `SubagentStart` 和 `SubagentStop` 上的必填字段。


* **Tool use ID（**，`str | None` / `string | undefined`**）：** 将同一个工具调用的 `PreToolUse` 和 `PostToolUse` 事件关联起来。
* **Context：** 在 TypeScript 中，包含用于取消操作的 `signal` 属性（`AbortSignal`）。在 Python 中，该参数保留供未来使用。



**Outputs**

回调函数（callback）会返回一个包含两类字段的对象：

* **顶层字段（Top-level fields）** 在每个事件上的作用均相同：`systemMessage` 用于向用户显示一条消息，而 `continue`（在 Python 中为 `continue_`）决定 Agent 在此 Hook 之后是否继续运行。
* **`hookSpecificOutput`** 用于控制当前操作。其内部的字段取决于具体的 Hook 事件类型。对于 `PreToolUse` Hook，你可以在此处设置 `permissionDecision`（`"allow"`、`"deny"`、`"ask"` 或 `"defer"`）、`permissionDecisionReason` 和 `updatedInput`。返回 `"defer"` 会终止当前查询，以便你可以在稍后**恢复它（resume it later）**。对于 `PostToolUse` Hook，你可以设置 `additionalContext` 来向工具结果追加信息。如果想在 Claude 看到工具输出之前对其进行替换，可以设置 `updatedToolOutput`（这在两个 SDK 中适用于任何工具）。较旧的 `updatedMCPToolOutput` 字段仅能替换 MCP 工具输出，现已被废弃（deprecated）。

返回 `{}` 表示允许执行该操作且不作任何更改。SDK 回调 Hook 使用与 **Claude Code shell 命令 Hook（Claude Code shell command hooks）** 相同的 JSON 输出格式，该文档记录了每个字段和特定事件的选项。

关于调用时的 **matcher**，简略知道它是一个filter, 每种 hook type的matcher不同:

Use matchers to filter when your callbacks fire. The matcher field matches against a different value depending on the hook event type. For example, tool-based hooks match against the tool name, while Notification hooks match against the notification type. 

详见claude agent sdk 文档。

hook回调函数 什么时候被调用？

只要在 hooks 参数中注册了这个函数，SDK 就会在对应事件触发时调用它 —— 但具体"什么时候"取决于你用 HookMatcher 注册时指定的 matcher（工具名过滤器）和事件类型（比如 PreToolUse）。

e.g. 

- case 1 不加matcher

```
hooks={
    "PreToolUse": [HookMatcher(hooks=[auto_approve_read_only])]  # 没写 matcher
}
```

这种情况下，只要触发 PreToolUse 事件（也就是任何工具被调用前），这个 hook 就会被执行——不管是 Read、Write、Bash 还是别的工具。这也解释了为什么函数内部要先手动判断 tool_name 是否在 read_only_tools 列表里，因为 matcher 层面没有帮你过滤。


- case 2 加了 matcher 限制

```
hooks={
    "PreToolUse": [HookMatcher(matcher="Read|Glob|Grep", hooks=[auto_approve_read_only])]
}
```

这种情况下，SDK 会先在框架层面做过滤，只有工具名匹配 Read、Glob 或 Grep 时才会调用这个函数。此时函数内部那行 if input_data["tool_name"] in read_only_tools 判断其实是"双保险"，理论上已经不会走到 else 分支了。
